In [0]:
%sql
create database if not exists ecommerce.dashboards

In [0]:
%sql
-- ── Customer 360 View — joins all prediction tables ──────────────────────
CREATE OR REPLACE VIEW ecommerce.dashboards.vw_customer_360 AS

SELECT
    cf.customer_unique_id,
    cf.customer_state,
    cf.customer_city,

    -- RFM
    cf.recency_days,
    cf.frequency,
    cf.monetary,
    cf.R_score,
    cf.F_score,
    cf.M_score,
    cf.RFM_score,
    cf.rfm_segment,

    -- Behavioural
    cf.avg_order_value,
    cf.avg_review_score,
    cf.late_order_rate,
    cf.customer_tenure_days,
    cf.active_months,
    cf.is_repeat_customer,

    -- Churn predictions
    cp.churn_probability,
    cp.churn_label_predicted,
    cp.risk_tier,
    cp.best_threshold,
    cp.prediction_date       AS churn_prediction_date,

    -- CLV predictions
    clv.predicted_clv_90d,
    clv.clv_tier,

    -- Segments
    cs.cluster_id,
    case cs.segment_name when "type_3" then "Dissatisfied High Spenders" when "type_2" then "Loyal Champions" when "type_1" then "Standard Buyers" end as segment_name,
    cs.centroid_distance,

    -- Revenue at risk (churn probability × predicted CLV)
    CAST(
        ROUND(cp.churn_probability * clv.predicted_clv_90d, 2)
    AS DOUBLE)               AS revenue_at_risk_90d

FROM ecommerce.gold.vw_customer_features    cf
LEFT JOIN ecommerce.gold.churn_predictions  cp
    ON cf.customer_unique_id = cp.customer_unique_id
    AND cp.prediction_date   = (
        SELECT MAX(prediction_date)
        FROM ecommerce.gold.churn_predictions
    )
LEFT JOIN ecommerce.gold.clv_predictions    clv
    ON cf.customer_unique_id = clv.customer_unique_id
    AND clv.prediction_date  = (
        SELECT MAX(prediction_date)
        FROM ecommerce.gold.clv_predictions
    )
LEFT JOIN ecommerce.gold.customer_segments  cs
    ON cf.customer_unique_id = cs.customer_unique_id;

In [0]:
%sql
select * from ecommerce.dashboards.vw_customer_360

In [0]:
%sql
use catalog ecommerce

In [0]:
import os

In [0]:
for i in spark.catalog.listDatabases():
    for j in spark.catalog.listTables(i.name):
        print(i.name+"."+j.name)
        if j.name.startswith("vw"):
            os.makedirs("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/DDL/view/"+i.name, exist_ok=True)
            with open("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/DDL/view/"+i.name+"/"+j.name+".sql","w") as f:
                f.write("-- Databricks notebook source\n")
                f.write(spark.sql(f"show create table {i.name}.{j.name}").collect()[0]['createtab_stmt'])
                f.close()
        else:
            os.makedirs("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/DDL/table/"+i.name, exist_ok=True)
            with open("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/DDL/table/"+i.name+"/"+j.name+".sql","w") as f:
                f.write("-- Databricks notebook source\n")
                f.write(spark.sql(f"show create table {i.name}.{j.name}").collect()[0]['createtab_stmt'])
                f.close()

In [0]:
import mlflow
import pandas as pd
import os

# Connect to your MLflow tracking server (already set in Databricks)
client = mlflow.tracking.MlflowClient()

# ── Step 1: Get all experiments ──────────────────────────────────────────────
experiments = client.search_experiments()

all_runs = []

# ── Step 2: Loop through each experiment and fetch all runs ──────────────────
for exp in experiments:
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        max_results=5000  # increase if needed
    )
    
    for run in runs:
        row = {
            "experiment_id":   exp.experiment_id,
            "experiment_name": exp.name,
            "run_id":          run.info.run_id,
            "run_name":        run.info.run_name,
            "status":          run.info.status,
            "start_time":      pd.to_datetime(run.info.start_time, unit="ms"),
            "end_time":        pd.to_datetime(run.info.end_time,   unit="ms"),
            "user_id":         run.info.user_id,
        }
        # Flatten metrics, params, tags
        row.update({f"metric_{k}": v for k, v in run.data.metrics.items()})
        row.update({f"param_{k}":  v for k, v in run.data.params.items()})
        row.update({f"tag_{k}":    v for k, v in run.data.tags.items()})
        
        all_runs.append(row)

# ── Step 3: Save to CSV ───────────────────────────────────────────────────────
df = pd.DataFrame(all_runs)
os.makedirs("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/MLFlow/", exist_ok=True)
df.to_csv("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/MLFlow/all_experiments_runs.csv", index=False)
print(f"Exported {len(df)} runs across {len(experiments)} experiments")

In [0]:
import os
import json
import sys

def path_to_dict(path):
    """
    Recursively generates a dictionary representing the folder structure.
    """
    # Initialize the dictionary for the current path
    hierarchy = {
        'name': os.path.basename(path),
        'path': os.path.abspath(path)
    }

    if os.path.isdir(path):
        hierarchy['type'] = 'folder'
        hierarchy['children'] = []
        
        # Iterate over directory contents
        for contents in os.listdir(path):
            full_path = os.path.join(path, contents)
            # Recursively call the function for subdirectories/files and append to children
            hierarchy['children'].append(path_to_dict(full_path))
    else:
        hierarchy['type'] = 'file'

    return hierarchy


structure = path_to_dict("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform")
print(json.dumps(structure, indent=2))


In [0]:
import os
import json
from typing import List, Dict

# Specify the root folder to search
root_folder = "/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/DDL"

results: List[Dict] = []

for dirpath, dirnames, filenames in os.walk(root_folder):
    folder_name = os.path.basename(dirpath)
    for filename in filenames:
        file_path = os.path.join(dirpath, filename)
        print(file_path)

In [0]:
for col in ["job_id", "job_name", "source_view", "target_table"]:
    results_df[col] = results_df[col].fillna("")

In [0]:
results_df.to_csv("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/jobs.csv", index=False)

In [0]:
spark.createDataFrame(results_df)